# Llama labeling run (Kaggle GPU)

Runs `pipeline/kaggle_llama_labeling.py` on this notebook's GPU, against code
and data you upload from the repo.

Nothing here is a copy of repo logic. The prompt, `text_for`, and the label
validation are the uploaded files themselves — in particular the prompt
version written into `model_meta` is parsed from the prompt file, so it cannot
drift the way a pasted-in prompt string would.

The config below is set for a **full 8,360-row run with prompt v4**, to
measure its corpus-wide `negative` yield against v3's. Swap `PROMPT` and
`OUTPUT_CSV` to relabel with a different version, or point `INPUT_CSV` at
`labeling_batch_gold300.csv` to score a prompt on the gold rows instead.

## Prepare the upload (run this locally, in the repo root)

```sh
rm -rf kaggle_upload
mkdir -p kaggle_upload/pipeline kaggle_upload/prompts
cp pipeline/__init__.py pipeline/labeling.py pipeline/eligibility.py \
   pipeline/kaggle_llama_labeling.py  kaggle_upload/pipeline/
cp evals/prompts/jiwon_llama_v4.md    kaggle_upload/prompts/
cp labeling_batch_2026-07-22.csv      kaggle_upload/
```

Upload `kaggle_upload/` as a **private** Kaggle dataset — the batch CSVs are
gitignored data and must not become public.

## Then, in this notebook

1. Settings → Accelerator → **GPU** (T4 x2 or P100).
2. Settings → Internet → **On** (pip and the model download need it).
3. Add Data → your private dataset.
4. Point `UPLOAD_DIR` below at it (right-hand panel shows the exact path), then
   Run All.

300 gold rows take roughly 2 minutes of labeling once the model is loaded, so
budget on the order of an hour for all 8,360 — watch the session time limit.

Kaggle's image already ships vLLM, so the `pip install` cell is usually a
no-op you can skip on a re-run.

In [ ]:
# --- config: the only cell you normally edit -------------------------------

# The attached dataset. Copy the exact path from the right-hand panel.
UPLOAD_DIR = "/kaggle/input/pnc-labeling"

# Paths inside the upload.
# v3 is the champion, but its corpus-wide `negative` yield collapsed (45 rows
# against ~234 implied by the human rate), leaving 30 negatives in the
# training set. v4 scores better on the negative axis alone (recall 0.786 vs
# 0.714 at comparable precision), so this run measures what v4 actually yields
# corpus-wide before deciding between them. See decision log 2026-08-09.
PROMPT = "prompts/jiwon_llama_v4.md"

INPUT_CSV = "labeling_batch_2026-07-22.csv"  # full corpus, 8,360 rows

OUTPUT_CSV = "/kaggle/working/labels_v4_full.csv"

# Writable staging dir: /kaggle/input is read-only, and the labeling module has
# to be importable as `pipeline.` from the working directory.
WORK_DIR = "/kaggle/working/run"

In [ ]:
import os
import shutil
from datetime import UTC, datetime

assert os.path.isdir(UPLOAD_DIR), (
    f"{UPLOAD_DIR} not found — fix UPLOAD_DIR. Attached: "
    f"{os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'nothing'}"
)

# Re-runnable: a previous partial run must not leave stale modules behind.
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(UPLOAD_DIR, WORK_DIR)
os.chdir(WORK_DIR)

# The four modules the labeling driver actually needs, resolved from imports.
REQUIRED = [
    "pipeline/__init__.py",
    "pipeline/labeling.py",
    "pipeline/eligibility.py",
    "pipeline/kaggle_llama_labeling.py",
    PROMPT,
    INPUT_CSV,
]
absent = [p for p in REQUIRED if not os.path.exists(p)]
assert not absent, f"missing from the upload: {absent}"

RUN_DATE = datetime.now(UTC).date().isoformat()
print("staged  :", WORK_DIR)
print("run date:", RUN_DATE)

In [ ]:
# Fail here, before the model download, rather than after it.
import csv

from pipeline.labeling import parse_prompt_version

with open(INPUT_CSV, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
missing = {"raw_item_id", "source", "title", "text_excerpt"} - set(rows[0])
assert not missing, f"input CSV is missing columns: {missing}"

with open(PROMPT, encoding="utf-8") as f:
    PROMPT_VERSION = parse_prompt_version(f.read())

print(f"{len(rows)} rows to label, prompt {PROMPT_VERSION}")

In [ ]:
# First run downloads the AWQ checkpoint — a few minutes.
!pip install -q vllm

In [ ]:
!python -m pipeline.kaggle_llama_labeling \
    --input "$INPUT_CSV" \
    --prompt "$PROMPT" \
    --output "$OUTPUT_CSV" \
    --run-date "$RUN_DATE"

In [ ]:
# Smoke test — run this before downloading anything.
import json
from collections import Counter

from pipeline.labeling import LABELS

with open(OUTPUT_CSV, encoding="utf-8") as f:
    out = list(csv.DictReader(f))

assert len(out) == len(rows), f"expected {len(rows)} labels, got {len(out)}"
assert all(r["label"] in LABELS for r in out), "invalid label in output"

meta = json.loads(out[0]["model_meta"])
assert meta["prompt_version"] == PROMPT_VERSION, (
    f"model_meta says {meta['prompt_version']}, prompt file says {PROMPT_VERSION}"
)

dist = Counter(r["label"] for r in out)
assert len(dist) > 1, f"degenerate output, everything is one class: {dist}"

print(json.dumps(meta, indent=2))
for label in LABELS:
    n = dist[label]
    print(f"{label:9} {n:6}  {n / len(out):6.1%}")

## Next

Download the output CSV from `/kaggle/working/`, then score it locally — the
human labels it is graded against live in the repo, not here:

```sh
python -m pipeline.quality_gate --gold-dir evals/items \
  --labels labels_v3_pilot.csv \
  --output evals/gate_report_v3_pilot.md --run-date <YYYY-MM-DD>
```

The report's **Acceptance criteria** table scores the run against the bar fixed
before it, and **Dev vs holdout** shows whether a gain came from the rows the
prompt was written against. Only relabel all 8,360 rows if the criteria clear.